# 1.BaseChatMemory运行流程解析

知识点讲解：Memory 运行流程与 LangChain 1.x 的记忆范式演进

1. LangChain 0.x 的 Memory 体系（已废弃）
   - 核心抽象：BaseMemory → BaseChatMemory → 各种具体 Memory 子类
   - BaseMemory 基类封装方法：memory_variables、load_memory_variables、aload_memory_variables、save_context、asave_context、clear、aclear
   - 两个核心方法（子类关键）：load_memory_variables(inputs) 链执行前调用，把历史注入链输入；save_context(inputs, outputs) 链执行后调用，写回本轮问答
   - 关键字段：chat_memory（内部持有的 BaseChatMessageHistory，真正的存储载体）、memory_variables（声明注入变量名）、input_key / output_key（从输入输出字典取对应值）、return_messages（True 返回消息列表，False 返回拼接纯文本）
   - 问题：Memory 隐式改写链的输入输出，行为不透明，难测试难调试

2. 衍生子类：SimpleMemory 与 BaseChatMemory
   - SimpleMemory：实现了记忆方法但不存储任何内容，可在"不需要记忆又不想改代码结构"时平替
   - BaseChatMemory：内置其他记忆组件的基类，针对聊天模型对话场景封装了历史

3. LangChain 1.x 的新范式（本示例采用）
   - 公式：记忆 = 消息历史存储（BaseChatMessageHistory）+ 链上显式注入
   - 存储层：InMemory / File / Redis 等，只管存取
   - 注入层：MessagesPlaceholder 占位，RunnableWithMessageHistory 自动读写
   - 裁剪层：trim_messages、SummarizationMiddleware
   - 好处：每一步显式可见、职责单一、易替换与单元测试

4. 新旧 API 对应关系（迁移对照表）
   旧写法                                      新写法
   memory.save_context({"input":q},{"output":a})  add_user_message(q)+add_ai_message(a)
   memory.load_memory_variables({})                chat_history.messages
   memory.clear()                                  chat_history.clear()
   memory.chat_memory                              chat_history 本身
   ConversationBufferWindowMemory                  trim_messages(strategy="last")
   ConversationSummaryBufferMemory                 SummarizationMiddleware
   ConversationEntityMemory                        with_structured_output 显式抽取

5. BaseChatMemory 为什么是抽象基类（源码要点）
   - memory_variables 与 load_memory_variables 是抽象方法，无默认实现
   - 其内部 save_context 本质就是 self.chat_memory.add_messages([HumanMessage, AIMessage])
   - clear 本质调用 self.chat_memory.clear()
   - 不同记忆策略注入的变量名与加载逻辑差异大，必须由子类各自定义，故不能直接实例化

6. 一轮问答对应两条消息
   - add_user_message + add_ai_message 合起来等价于旧版一次 save_context
   - 顺序重要：先 user 后 ai，否则对话顺序错乱
   - 批量写法 add_messages([HumanMessage(...), AIMessage(...)]) 更高效，持久化下可合并两次 IO 为一次

7. 历史在链上的三种注入方式
   - 手动注入：chain.invoke({"query": q, "chat_history": chat_history.messages})
   - 自动读写：RunnableWithMessageHistory 按 session_id 自动存取（推荐）
   - Agent 场景：LangGraph 的 checkpointer 按 thread_id 持久化全部状态

8. clear() 的使用场景
   - 用户点"新建对话"重置上下文；会话超时或换话题时主动清理
   - 注意：对持久化实现（FileChatMessageHistory）clear 会真正删除磁盘数据

9. 注意事项
   - messages 返回的是消息对象列表而非字符串，不要直接拼进 f-string
   - 该组件不做任何自动裁剪，长对话必须配合 trim_messages 或摘要
   - 多用户场景务必按 session_id / user_id 隔离历史，防止记忆串号


In [9]:
from langchain_core.chat_history import InMemoryChatMessageHistory
chat_history = InMemoryChatMessageHistory()
print("保存前:", chat_history.messages)
chat_history.add_user_message("你好，我是慕小课你是谁")
chat_history.add_ai_message("你好，我是DeepSeek,有什么可以帮到您的")

保存前: []


In [10]:
print("保存后:", chat_history.messages)

保存后: [HumanMessage(content='你好，我是慕小课你是谁', additional_kwargs={}, response_metadata={}), AIMessage(content='你好，我是DeepSeek,有什么可以帮到您的', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


In [11]:
chat_history.clear()
print("清空后:", chat_history.messages)

清空后: []
